# VL12 – Local Large Language Models

In this lab, we explore the deployment of **local large language models** on **resource-constrained hardware**.

We analyze:
- **resource utilization** (memory, latency, throughput)
- the impact of **model size** and **quantization precision**
- **task performance** on an **extractive Question & Answering** task

The goal is to understand conceptually and practically the challenges of deploying LLMs, and the tradeoffs that come with it.

## 1. Setup the environment

### 1.1 Install Ollama & libraries

#### Install ollama
Go the the course repository, and run the provided script

````bash
bash script/install_ollama.sh
source ~/.bashrc
````

#### Verify installation
````bash
which ollama
ollama --version
````

#### Install Python library
````bash
pip install ollama
````

### 1.2 Download models

#### Start Ollama server
````bash
export OLLAMA_HOST=127.0.0.1:11435
ollama serve
````

#### Download model in another terminal
````bash
export OLLAMA_HOST=127.0.0.1:11435
ollama pull llama3.2:3b
ollama pull llama3.2:3b-instruct-q8_0
ollama pull llama3.1:8b-instruct-q4_K_M
````


## 2. Checking resource utilization

#### Why are we doing this?

Until now, we have treated language models as algorithms that receive text and generate predictions.

In practice, however, every model also consumes **computational resources**. Before comparing model quality, it is important to understand **how local LLMs interact with the available hardware**.

In this section we will observe a model while it is running and answer the following questions:

- When is GPU memory allocated?
- When is the GPU actually performing computations?
- Does memory disappear after generation finishes?
- How do different models affect hardware utilization?

Rather than relying on theory alone, we will measure these effects directly on the GPU.

Because GPUs are only accessible inside your Jupyter job, all commands below must be executed from terminals opened **inside this notebook environment**.

You will use two terminals:
- one for monitoring resources
- one for running model prompts

#### Terminal 1 - Monitoring GPU resources
Open the terminal (File -> New -> Terminal) and run:
````bash
watch nvidia-smi
````
This continuously shows GPU status and updates every second.

#### Terminal 2 - Running prompts
Open a second terminal and run:
````bash
export OLLAMA_HOST=127.0.0.1:11434
ollama run llama3.2:3b "Write a 150-word explanation of transformers."
````
Leave the monitoring terminal visible while the model is running.

### 2.1 What resources metrics to observe
`nvidia-smi` reports many GPU statistics. We will focus on a small set of metrics that are particularly useful for understanding LLM inference.

![nvidia-smi metrics](img/nvidia-smi-metrics.png)

### 2.2 GPU devices and memory usage

After starting `watch nvidia-smi`, run the prompt in the second terminal.

You should observe something similar to:

![nvidia-smi metrics](img/nvidia-smi.png)

What to focus on:

- Processes section
    - Look for the ollama process
    - Note how much GPU memory (MB) it uses
- Memory usage
    - Model memory is allocated when the model loads
    - Memory usually does not drop immediately after output finishes
- Multiple GPUs
    - Only one GPU / MIG slice is actually used by your job
    - Others remain mostly idle

> Key insight? Loading a model consumes memory even when it is not generating tokens.

### 2.3 Observing compute spikes over time
To understand **when the GPU is actually computing**, switch the monitoring terminal to

````bash
nvidia-smi dmon -s pucm
````

You can also filter by a gpu in particular. For example, to monitor GPU 0 only:

````bash
nvidia-smi dmon -i 0 -s pucm
````

We will see something similar to: 

![nvidia-smi metrics](img/nvidia-smi-dmon.png)

Focus on the following metrics:

- **pwr (W)**
    - GPU power consumption
    - Typically spikes while tokens are generated

- **pclk (MHz)**
    - GPU clock frequency
    - Increases during active computation

- **fb (MB)**
    - GPU memory usage
    - Usually remains stable after the model has loaded

Notice the difference between **loading** and **inference**:

- Loading the model mainly allocates GPU memory.
- During inference, GPU utilization and power increase as the model predicts each new token.
- Once generation finishes, computation stops, but the model often remains loaded in memory.

### 2.4 Let's compare model footprints

Let's repeat the previous steps, running prompts with different models

````bash
ollama run llama3.2:3b-instruct-q8_0 "Write a 150-word explanation of transformers."
````

````bash
ollama run llama3.1:8b-instruct-q4_K_M "Write a 150-word explanation of transformers."
````

Compare your observations.

- Which model used the most GPU memory?
- Which model appeared to generate text faster?
- Did the loading behaviour change?
- Which differences do you think are caused by **model size**, and which by **quantization**?

These observations will help us understand the trade-offs between model capability, memory usage, and inference performance under hardware constraints.

### 2.5 Managing loaded models with Ollama

Ollama keeps recently used models loaded in memory to reduce loading time for future requests. You can inspect which models are currently loaded and how much GPU memory they occupy.

```bash
ollama ps
```

The output shows:

- the loaded model(s)
- whether they are running on the GPU or CPU
- how much memory they occupy
- how long they will remain loaded before being automatically unloaded

If you are running low on GPU memory, you can manually unload a model:

```bash
ollama stop <model>
```

For example:

```bash
ollama stop llama3.2:3b
```

Run `ollama ps` again and observe that the model disappears from the list. You should also see the GPU memory being released in `nvidia-smi`.


## 3. Using LLMs in Python

So far, we have interacted with LLMs from the terminal.  
In practice, however, LLMs are usually **used as part of an application**, not as a chat interface.

Ollama exposes a **local API**, which allows Python programs to send prompts to a model and receive generated text.

Before using Python, make sure that:
- the Ollama server is **running** (`ollama serve`)
- it is reachable at the local address used in this notebook

In the next steps, we will:
1. configure the Python environment to talk to the Ollama server  
2. send prompts from Python code  
3. measure response time and resource usage during inference


### 3.1 Setup environment 

In [ ]:
import os
os.environ["OLLAMA_HOST"] = "http://127.0.0.1:11435"  # change to your server
import ollama

Let's check that everything works. Let's list the available models

In [ ]:
import pandas as pd

models = ollama.list()

df = pd.DataFrame(models["models"])
df.head()

### 3.2 Generating text with Ollama in Python

Ollama provides **three main interfaces** for text generation in Python.  
They differ in how results are returned and how much control you have during generation.


#### Simple generation (blocking)
This is the simplest interface.  
The call blocks until the full response is generated.

In [ ]:
resp = ollama.generate(
    model="llama3.2:3b",
    prompt="Say hello in one short sentence."
)
print(resp["response"])


#### Streaming generation (token-by-token)
This interface returns a stream of partial outputs as the model generates text.

In [ ]:
stream = ollama.generate(
    model="llama3.2:3b",
    prompt="Write a 150-word explanation of transformers.",
    stream=True
)

text = ""
for chunk in stream:
    piece = chunk.get("response", "")
    text += piece
    print(piece, end="", flush=True)

print("\n\n---\nDone.")

#### Chat-style interaction (messages)
This interface uses role-based messages, similar to chat APIs.
Instead of sending a single text prompt, we send a list of **messages**, each with a role.

**Message structure**

Each message is a dictionary with two fields:

- role
Defines who is speaking:

    - system → instructions that set behavior or style
    - user → user input or questions
    - assistant → previous model responses (optional)
- content
The actual text of the message.


**Using conversation history as context**

The model does not remember past interactions automatically.
To give it context, you must explicitly include the conversation history in the messages list.

In [ ]:
resp = ollama.chat(
    model="llama3.2:3b",
    messages=[
        {"role": "system", "content": "You are concise."},
        {"role": "user", "content": "Explain tokenization in one sentence."}
    ]
)
print(resp["message"]["content"])


#### Chat-sylte interaction with streaming

In [ ]:

stream = ollama.chat(
    model="llama3.2:3b",
    messages=[
        {"role": "system", "content": "You are concise."},
        {"role": "user", "content": "Explain tokenization in one short paragraph."},
    ],
    stream=True,
)

for chunk in stream:
    # Each chunk is a dict; streamed text arrives in chunk["message"]["content"]
    content = chunk.get("message", {}).get("content", "")
    print(content, end="", flush=True)

print()  # newline at end


## 4. Comparing models

Now that we know how to run prompts from Python, we can **compare different models under the same conditions**.

The goal of this section is **not** to judge which model gives the “best answer”, but to understand how **model choice affects performance and resource usage**.

We will use a helper function called `generate_with_metrics`.  
It takes:
- a model name
- a prompt
- generation options (the same ones used by the Ollama API)

and returns both the model output and a set of **performance metrics**.


**Generation metrics**

The following metrics are collected for each run:

- **`wall_s`**  
  Total wall-clock time for the request.  
  Includes prompt processing, token generation, and system overhead.

- **`ttft_s`** *(streaming only)*  
  *Time to First Token*.  
  Measures how quickly the model starts responding, which affects perceived latency.

- **`prompt_tokens`**  
  Number of tokens in the input prompt.

- **`gen_tokens`**  
  Number of tokens generated by the model.

- **`prompt_tok_s`**  
  Prompt processing speed (tokens per second).  
  Mostly affected by model size and hardware.

- **`gen_tok_s`**  
  Generation speed (tokens per second).  
  This is the main throughput metric for inference.

In [ ]:
import time

def generate_with_metrics(model, prompt, stream=False, print_live=True, **kwargs):
    t0 = time.time()

    text = ""
    first_token_time = None
    last = None  # we'll keep the last response/chunk (often contains stats)

    if stream:
        for chunk in ollama.generate(model=model, prompt=prompt, stream=True, **kwargs):
            if first_token_time is None:
                first_token_time = time.time()

            piece = chunk.get("response", "")
            if piece:
                text += piece
                if print_live:
                    print(piece, end="", flush=True)

            last = chunk

        if print_live:
            print()
    else:
        last = ollama.generate(model=model, prompt=prompt, stream=False, **kwargs)
        text = last.get("response", "")

    t1 = time.time()

    # Stats (sometimes present depending on server/version)
    prompt_tokens = last.get("prompt_eval_count")
    gen_tokens = last.get("eval_count")

    prompt_ns = last.get("prompt_eval_duration")  # nanoseconds
    gen_ns = last.get("eval_duration")            # nanoseconds

    prompt_s = (prompt_ns / 1e9) if prompt_ns else None
    gen_s = (gen_ns / 1e9) if gen_ns else None

    metrics = {
        "wall_s": t1 - t0,
        "ttft_s": (first_token_time - t0) if first_token_time else None,
        "prompt_tokens": prompt_tokens,
        "gen_tokens": gen_tokens,
        "prompt_tok_s": (prompt_tokens / prompt_s) if (prompt_tokens and prompt_s) else None,
        "gen_tok_s": (gen_tokens / gen_s) if (gen_tokens and gen_s) else None,
    }

    return text, metrics

### 4.1 Compare metrics
No we run some comparisions, with the three models we have downloaded. To make a fair comparison, consider the following:

  - First run includes cold-start overhead
  - Use same prompt and `num_predict` for fair comparison
  - Missing values mean the server did not report them

In [ ]:
text, m = generate_with_metrics(
    "llama3.2:3b",
    "Write a 100-word explanation of word embeddings.",
    #stream=True,
    options={"temperature": 0.2, "num_predict": 180},
)
print("\n\nMetrics:", m)

In [ ]:
text, m = generate_with_metrics(
    "llama3.2:3b-instruct-q8_0",
    "Write a 100-word explanation of word embeddings.",
    #stream=True,
    options={"temperature": 0.2, "num_predict": 180},
)
print("\n\nMetrics:", m)

### 4.3 Reflection
What did we learn about the resource utilization of the models? 

## 5. Extractive Question & Answering with LLMs

So far, we have focused mainly on **resource utilization**: memory usage, latency, and throughput.  
In this section, we shift focus to **task performance**.


### 5.1 What is extractive Question & Answering?

In **extractive Question & Answering (Q&A)**, the model is given:
- a **context passage** (text)
- a **question**

The goal is to **extract the answer directly from the context**, rather than generating a free-form response.

Key characteristics:
- The answer must appear **verbatim** in the context
- No new information should be invented
- Performance can be evaluated objectively

This makes extractive Q&A a good benchmark task.


#### Comparison dimensions

We vary two key factors:

- **Number of parameters**
  - Smaller models are cheaper to run
  - Larger models may capture more complex patterns

- **Quantization level**
  - Lower-bit quantization → lower memory usage and faster inference
  - Higher-bit quantization → potentially better accuracy


#### Models used in this experiment

We will compare the following models:

- `llama3.2:3b`(-instruct-q4_0)    
    Small, general-purpose model

- `llama3.2:3b-instruct-q8_0`  
  Same parameter count, higher-precision quantization, instruction-tuned

- `llama3.1:8b-instruct-q4_K_M`  
  Larger model with more aggressive quantization

These models allow us to isolate:
- the effect of **model size**
- the effect of **quantization**
- the interaction between both under hardware constraints

### 5.2 Downloading and loading the dataset
We will use examples from **SQuAD (Stanford Question Answering Dataset)**.

SQuAD provides:
- a short context paragraph
- a question about that paragraph
- a gold-standard answer span

SQuAD was originally designed for **task-specific Q&A models**, not for general-purpose LLMs.  
This makes it interesting to test how well modern LLMs perform on a task they were not explicitly trained for.

To download the model, open the terminal, go to the root of our repository and run:

````bash
mkdir -p data/squad

curl -L \
  https://rajpurkar.github.io/SQuAD-explorer/dataset/dev-v2.0.json \
  -o data/squad/dev-v2.0.json
````

### 5.3 Prepare the dataset
The code below converts the json to a pandas dataframe.

In [ ]:
import json

PATH="../../data/squad/dev-v2.0.json"

with open(PATH, "r", encoding="utf-8") as f:
    squad = json.load(f)

rows = []
for article in squad["data"]:
    title = article.get("title", "")
    for para in article["paragraphs"]:
        context = para["context"]
        for qa in para["qas"]:
            qid = qa["id"]
            question = qa["question"]
            is_impossible = qa.get("is_impossible", False)
            answers = qa.get("answers", [])  # list of {"text":..., "answer_start":...}

            # gold answers (texts). For unanswerable, keep empty list (we handle later)
            gold_texts = [a["text"] for a in answers]

            rows.append({
                "id": qid,
                "title": title,
                "context": context,
                "question": question,
                "is_impossible": is_impossible,
                "gold_answers": gold_texts,
            })

print("Examples:", len(rows))
print("Sample row keys:", rows[0].keys())
print("Unanswerable count:", sum(r["is_impossible"] for r in rows))

# Optional: put into a pandas DataFrame (nice for inspection)
import pandas as pd

df = pd.DataFrame(rows)
df.head(3)

### 5.4 Task performance
We already defined the function `generate_with_metrics` that can provide some performance metrics. Below we define some task-specific metrics.


We evaluate extractive Question & Answering using **SQuAD-style metrics**, which compare the model’s predicted answer span against one or more gold answers.

#### Span-level quality metrics

- **Exact Match (EM)**  
  Measures whether the predicted answer **exactly matches** a gold answer  
  (after normalization: lowercase, no punctuation, no articles).

  - EM = 1 → exact span match  
  - EM = 0 → any mismatch

- **Token-level F1**  
  Measures **overlap between tokens** in the prediction and the gold answer.

  - Precision: how much of the prediction is correct  
  - Recall: how much of the gold answer is recovered  
  - F1 balances both

F1 is more forgiving than EM and rewards **partially correct spans**.

When multiple gold answers exist, we take the **maximum EM and F1** over all of them (standard SQuAD practice).


#### Unanswerable questions (SQuAD v2)

Some questions are **not answerable** from the given context.

- Gold label: `NOT ANSWERABLE`
- The model is instructed to output exactly this string when no answer exists

For these cases:
- EM/F1 are computed against `NOT ANSWERABLE`
- Correct abstention counts as a correct decision


#### Behavior metrics (computed separately)

To keep task performance and behavior distinct, we also track:

- **Hallucination rate**  
  Fraction of unanswerable questions where the model **produces an answer anyway**

- **False abstention rate**  
  Fraction of answerable questions where the model incorrectly replies `NOT ANSWERABLE`

- **Answer attempt rate**  
  Fraction of questions where the model attempts an answer at all

These metrics capture **decision behavior**, not span quality, and are analyzed separately from EM/F1.

In [ ]:
import re, string
from collections import Counter

def normalize(s: str) -> str:
    s = s.lower()
    s = re.sub(r"\b(a|an|the)\b", " ", s)
    s = s.translate(str.maketrans("", "", string.punctuation))
    s = " ".join(s.split())
    return s  

def f1_score(pred: str, gold: str) -> float:
    pred_toks = normalize(pred).split()
    gold_toks = normalize(gold).split()
    if not pred_toks and not gold_toks:
        return 1.0
    if not pred_toks or not gold_toks:
        return 0.0
    common = Counter(pred_toks) & Counter(gold_toks)
    num_same = sum(common.values())
    if num_same == 0:
        return 0.0
    precision = num_same / len(pred_toks)
    recall = num_same / len(gold_toks)
    return 2 * precision * recall / (precision + recall)

def exact_match(pred: str, gold: str) -> float:
    return 1.0 if normalize(pred) == normalize(gold) else 0.0

def squad_metrics(pred: str, gold_answers, is_impossible: bool):
    # For unanswerable questions, treat gold as a single label:
    if is_impossible:
        gold_answers = ["NOT ANSWERABLE"]

    # If there are multiple gold answers, SQuAD takes max over them
    em = max(exact_match(pred, g) for g in gold_answers) if gold_answers else 0.0
    f1 = max(f1_score(pred, g) for g in gold_answers) if gold_answers else 0.0
    return em, f1

def normalize_pred(s):
    s = s.strip().upper()
    s = s.translate(str.maketrans("", "", string.punctuation))
    return s    
  
def behavior_metrics(results_df):
    """
    Computes decision-level behavior metrics:
    - hallucination_rate
    - false_abstention_rate
    - answer_attempt_rate
    """
    unanswerable = results_df["is_impossible"]
    answerable = ~results_df["is_impossible"]

    norm_pred = results_df["pred"].apply(normalize_pred)

    hallucinations = (unanswerable) & (norm_pred != "NOT ANSWERABLE")
    false_abstain = (answerable) & (norm_pred == "NOT ANSWERABLE")
    answered = norm_pred != "NOT ANSWERABLE"

    return {
        "hallucination_rate": hallucinations.sum() / unanswerable.sum()
            if unanswerable.sum() > 0 else 0.0,
        "false_abstention_rate": false_abstain.sum() / answerable.sum()
            if answerable.sum() > 0 else 0.0,
        "answer_attempt_rate": answered.sum() / len(results_df)
            if len(results_df) > 0 else 0.0,
    }    

### 5.5 Running the task
The helpers below, help us execute the task. Here, `run_task` executes the task over multiple examples, calls the model, and computes:
  - SQuAD span metrics (EM, F1)
  - decision behavior metrics
  - performance and resource metrics

In [ ]:
def make_prompt(context, question):
    return f"""Answer the question using ONLY words from the context.
If the question cannot be answered from the context, reply exactly: NOT ANSWERABLE.
Reply with only the answer text. 

Context:
{context}

Question:
{question}

Answer:"""


In [ ]:
row = rows[0]
make_prompt(row["context"], row["question"])

In [ ]:
def run_task(model, data, n=50, stream=False, options=None):
    results = []
    options = options or {"temperature": 0.0, "num_predict": 64}

    for i, row in enumerate(data[:n]):
        prompt = make_prompt(row["context"], row["question"])

        pred, perf = generate_with_metrics(
            model=model,
            prompt=prompt,
            stream=stream,
            print_live=False,
            options=options
        )

        pred = pred.strip()
        em, f1 = squad_metrics(pred, row["gold_answers"], row["is_impossible"])

        results.append({
            "id": row["id"],
            "question": row["question"],
            "context": row["context"],
            "gold_answers": row["gold_answers"],
            "is_impossible": row["is_impossible"],
            "pred": pred,
            "em": em,
            "f1": f1,
            **perf
        })

        if (i + 1) % 10 == 0:
            print(f"{i+1}/{n} done")

    results = pd.DataFrame(results)

    behavior = behavior_metrics(results)

    # Aggregate summary
    summary = {
        "model": model,
        "n": len(results),

        # quality (span metrics)
        "EM": results["em"].mean(),
        "F1": results["f1"].mean(),

        # behavior (decision-level)
        **behavior,

        # performance
        "avg_wall_s": results["wall_s"].mean(),
        "avg_prompt_tokens": results["prompt_tokens"].dropna().mean(),
        "avg_gen_tokens": results["gen_tokens"].dropna().mean(),
        "avg_prompt_tok_s": results["prompt_tok_s"].dropna().mean(),
        "avg_gen_tok_s": results["gen_tok_s"].dropna().mean(),
    }

    return results, summary

### 5.6 Compare models on task performance 

In [ ]:
model_name = "llama3.2:3b"  # or your q8 tag, etc.

res_llama3b_q4, summary = run_task(
    model=model_name,
    data=rows,
    n=100,
    stream=False,
    options={"temperature": 0.0, "num_predict": 64}
)
summary

In [ ]:
model_name = "llama3.2:3b-instruct-q8_0"

res_llama3b_q8, summary = run_task(
    model=model_name,
    data=rows,
    n=100,
    stream=False,
    options={"temperature": 0.0, "num_predict": 64}
)
summary

In [ ]:
model_name = "llama3.1:8b-instruct-q4_K_M"

res_llama8b_q4, summary = run_task(
    model=model_name,
    data=rows,
    n=100,
    stream=False,
    options={"temperature": 0.0, "num_predict": 64}
)
summary

### 5.7 Inspect the errors

In [ ]:
def inspect_errors(res, k=5):
    print("A) Unanswerable questions where the model answered anyway")
    print("-" * 60)
    a = res[(res["is_impossible"]) & (res["pred"] != "NOT ANSWERABLE")]
    display(a[["question", "gold_answers", "pred", "f1"]].head(k))

    print("\nB) Answerable questions where the model abstained")
    print("-" * 60)
    b = res[(~res["is_impossible"]) & (res["pred"] == "NOT ANSWERABLE")]
    display(b[["question", "gold_answers", "pred", "f1"]].head(k))

    print("\nC) Lowest non-zero F1 (almost-right answers)")
    print("-" * 60)
    c = res[(res["f1"] > 0.0) & (res["f1"] < 1.0)].sort_values("f1")
    display(c[["question", "gold_answers", "pred", "f1"]].head(k))


In [ ]:
inspect_errors(res_llama8b_q4, k=5)

## 6. Reflection

Based on the extractive Q&A experiments, reflect on the following questions.  
Focus on **task performance in relation to model size and quantization**.

- How did **task performance** change when moving from smaller to larger models?
  - Were the improvements consistent across examples?

- What was the effect of **quantization precision** on task performance?
  - Did higher-precision quantization (e.g. `q8`) noticeably improve results?

- How do:
  - a **larger model with lower precision**  
    (`llama3.1:8b-instruct-q4_K_M`)
  - compare to a **smaller model with higher precision**  
    (`llama3.2:3b-instruct-q8_0`)?
